In [7]:
import os
import time
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
def encoded_pixels_to_masks(fname: str, df: pd.DataFrame):
    fname_df = df[df['ImageId'] == fname]
    masks = np.zeros((256 * 1600, 4), dtype=int)

    for i_row, row in fname_df.iterrows():
        cls_id = row['ClassId']
        encoded_pixels = row['EncodedPixels']
        if encoded_pixels is not np.nan:
            pixel_list = list(map(int, encoded_pixels.split(' ')))
            for i in range(0, len(pixel_list), 2):
                start_pixel = pixel_list[i] - 1
                num_pixel = pixel_list[i + 1]
                masks[start_pixel:(start_pixel + num_pixel), cls_id - 1] = 1

    masks = masks.reshape(256, 1600, 4, order='F')
    
    return masks


def masks_to_encoded_pixels(single_mask, target_class, thresholds):
    threshold_value = thresholds[target_class]
    binary_mask = (single_mask.flatten(order='F') > threshold_value).astype(int)
    
    binary_mask = np.concatenate(([0], binary_mask, [0]))
    
    transition_indices = np.where(binary_mask[:-1] != binary_mask[1:])[0] + 1
    lengths = transition_indices[1::2] - transition_indices[::2]
    starts = transition_indices[::2]
    
    return ' '.join(f"{start} {length}" for start, length in zip(starts, lengths))

In [10]:
class SeverstalSteelDataset(Dataset):
    def __init__(self, fnames, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.fnames = fnames

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        img_id = self.fnames[idx]
        img_path = os.path.join(self.img_dir, img_id)
        img = np.array(Image.open(img_path).convert('RGB'))
        masks = encoded_pixels_to_masks(img_id, self.df)
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        masks = torch.tensor(masks, dtype=torch.float32).permute(2, 0, 1)
        
        return img_id, img, masks


def collate_fn(batch_items):
    filenames = [batch[0] for batch in batch_items]
    images_batch = torch.stack([batch[1] for batch in batch_items])
    masks_batch = torch.stack([batch[2] for batch in batch_items])
    
    return filenames, images_batch, masks_batch

In [11]:
class SegModel(torch.nn.Module):
    def __init__(self):
        super(SegModel, self).__init__()
        self.model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet',classes=4)
    def forward(self, x):
        return self.model(x)

In [12]:
def load_data(train_val_csv, test_csv, train_val_img_dir, test_img_dir, max_samples=None):
    train_val_df = pd.read_csv(train_val_csv)
    train_val_fnames = pd.unique(train_val_df.ImageId)
    test_df = pd.read_csv(test_csv)
    test_fnames = pd.unique(test_df.ImageId)
    
    train_fnames, val_fnames = train_test_split(train_val_fnames, test_size=0.25, random_state=12)
    
    if max_samples:
        train_fnames = train_fnames[:max_samples]
        val_fnames = val_fnames[:max_samples]
        test_fnames = test_fnames[:max_samples]
    
    train_dataset = SeverstalSteelDataset(train_fnames, train_val_df, train_val_img_dir)
    val_dataset = SeverstalSteelDataset(val_fnames, train_val_df, train_val_img_dir)
    test_dataset = SeverstalSteelDataset(test_fnames, test_df, test_img_dir)
    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
    
    return train_loader, val_loader, test_loader

In [13]:
thresholds = [0.5, 0.5, 0.5, 0.5]
def init_model(lr, momentum):
    model = SegModel()
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    return model, criterion, optimizer

In [14]:
def train(model, loss_function, optimizer, data_loader, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        total_loss = 0.0
        
        for filenames, images, masks in data_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            predictions = model(images)
            loss = loss_function(predictions, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        print(f"Epoch {epoch + 1}/{num_epochs} - Average Loss: {avg_loss:.4f}")
        print(f"Time taken for epoch: {time.time() - epoch_start_time:.2f} seconds")
    
    return model

In [15]:
def evaluate(model, data_loader):  
    model.eval()
    all_dice_scores = []

    with torch.no_grad():
        for batch in data_loader:
            file_names, images, true_masks = batch

            images = images.to(device)
            true_masks = true_masks.to(device)

            logits = model(images)
            probabilities = torch.sigmoid(logits)

            predictions = probabilities.cpu().numpy()
            true_masks = true_masks.cpu().numpy()

            for file_idx, file_name in enumerate(file_names):
                for cls_idx in range(4):
                    thresholded_prediction = (predictions[file_idx, cls_idx] > thresholds[cls_idx]).astype(int)
                    dice_score = get_dice_coefficient(thresholded_prediction, true_masks[file_idx, cls_idx])
                    all_dice_scores.append((cls_idx, dice_score))

    avg_dice_score = np.mean([score for _, score in all_dice_scores])
    print(f"Average dice score: {avg_dice_score:.4f}")

    for cls_idx in range(4):
        class_scores = [score for class_id, score in all_dice_scores if class_id == cls_idx]
        avg_class_score = np.mean(class_scores) if class_scores else 0.0
        print(f"Dice score for class {cls_idx + 1}: {avg_class_score:.4f}")

In [16]:
def get_dice_coefficient(prediction, target):
    prediction_flat = prediction.flatten()
    target_flat = target.flatten()
    intersection = (prediction_flat * target_flat).sum()
    total = prediction_flat.sum() + target_flat.sum()
    if total == 0:
        return 1.0
    
    dice_score = 2.0 * intersection / total
    return dice_score

In [26]:
train_loader, val_loader, test_loader = load_data('train.csv','sample_submission.csv','train_images','test_images',max_samples=50000)

In [27]:
model, criterion, optimizer = init_model(lr=0.005, momentum=0.95)
model.to(device)

SegModel(
  (model): Unet(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchN

In [28]:
train_epochs = 6
train(model, criterion, optimizer, train_loader, train_epochs)

Epoch 1/6 - Average Loss: 0.0435
Time taken for epoch: 330.38 seconds
Epoch 2/6 - Average Loss: 0.0274
Time taken for epoch: 306.56 seconds
Epoch 3/6 - Average Loss: 0.0233
Time taken for epoch: 306.98 seconds
Epoch 4/6 - Average Loss: 0.0207
Time taken for epoch: 306.93 seconds
Epoch 5/6 - Average Loss: 0.0186
Time taken for epoch: 306.72 seconds
Epoch 6/6 - Average Loss: 0.0178
Time taken for epoch: 306.99 seconds


SegModel(
  (model): Unet(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchN

In [29]:
model.load_state_dict(torch.load('weights_' + str(train_epochs) + '.pth', weights_only=True))

<All keys matched successfully>

In [30]:
train_val_csv = 'train.csv'

df = pd.read_csv(train_val_csv)

df.ImageId

0       0002cc93b.jpg
1       0007a71bf.jpg
2       000a4bcdd.jpg
3       000f6bf48.jpg
4       0014fce06.jpg
            ...      
7090    ffcf72ecf.jpg
7091    fff02e9c5.jpg
7092    fffe98443.jpg
7093    ffff4eaa8.jpg
7094    ffffd67df.jpg
Name: ImageId, Length: 7095, dtype: object

In [31]:
img_id = 'ffff4eaa8.jpg'
df[df.ImageId==img_id]

,ImageId,ClassId,EncodedPixels
7093,ffff4eaa8.jpg,3,16899 7 17155 20 17411 34 17667 47 17923 60 18...


In [32]:
evaluate(model, val_loader)

Average dice score: 0.8565
Dice score for class 1: 0.8758
Dice score for class 2: 0.9616
Dice score for class 3: 0.6535
Dice score for class 4: 0.9351


In [34]:
model.eval()
final_submission = []

with torch.no_grad():
    for file_names, images, _ in test_loader:
        images = images.to(device)
        
        predictions = model(images)
        probabilities = torch.sigmoid(predictions).cpu().numpy()

        for idx, file_name in enumerate(file_names):
            class_encoded_pixels = []

            for class_index in range(4):
                encoded_pix = masks_to_encoded_pixels(
                    probabilities[idx, class_index], 
                    class_index, 
                    thresholds
                )
                class_encoded_pixels.append((file_name, class_index + 1, encoded_pix))

            final_submission.extend(class_encoded_pixels)

submission_dataframe = pd.DataFrame(
    final_submission, 
    columns=['ImageId', 'ClassId', 'EncodedPixels']
)

submission_dataframe.to_csv('submission.csv', index=False)